# State Reducers [Step 4 - Replacement vs Accumulation]

> **MLCourse - Agentic AI - LangGraph**

This notebook explains how state updates work in LangGraph.
By default, node returns **replace** state fields. With `Annotated`
types and reducer functions like `operator.add`, nodes can
**accumulate** values instead. This is critical for message histories
and any list-based state.

### Import core libraries


In [ ]:
import os                          # Environment variable access
import operator                    # operator.add for list accumulation
from typing import TypedDict, Annotated, Sequence  # Type annotations
from dotenv import load_dotenv     # Load API keys from .env

load_dotenv()                      # Initialize .env loading


### API key guard -- ChatOllama is local, always green


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")


### Part 1: Default Behavior -- Replacement


In [ ]:
# When a node returns `{"field": new_value}`, the new value
# **replaces** the old value entirely. This is the default.

from typing_extensions import TypedDict

class SimpleState(TypedDict):
    messages: list                 # A list of messages
    count: int                     # A counter

# Simulate two nodes that each return partial updates.
# With replacement, the second node's list overwrites the first.

def add_one_message(state: SimpleState) -> dict:
    """Node 1: Returns a list with one message."""
    return {"messages": ["Hello from node 1"]}

def add_another_message(state: SimpleState) -> dict:
    """Node 2: Returns a list with one message (replaces, does not append)."""
    return {"messages": ["Hello from node 2"]}

from langgraph.graph import StateGraph, START, END

# Build a simple 2-node graph to demonstrate replacement
graph = StateGraph(SimpleState)
graph.add_node("node1", add_one_message)
graph.add_node("node2", add_another_message)
graph.add_edge(START, "node1")
graph.add_edge("node1", "node2")
graph.add_edge("node2", END)

app_replacement = graph.compile()

# Run it -- notice that node2 REPLACES the messages list
result = app_replacement.invoke({"messages": [], "count": 0})
print("After replacement (default behavior):")
print("  Messages:", result["messages"])
print("  Only the LAST node's list survives!")
print()

# This is usually NOT what you want for message histories.
# You want messages to ACCUMULATE. That is where reducers come in.


### Part 2: Reducers with Annotated Types


In [ ]:
# A reducer is a function that combines the old state value with the
# new value returned by a node. We declare reducers using
# `Annotated[Type, reducer_function]` in the TypedDict.
#
# Common reducer: `operator.add` -- concatenates lists.

# Define a state with a reducer on the messages field.
# Annotated[list, operator.add] means: when a node returns a list
# for "messages", concatenate it with the existing list.

class AccumulatingState(TypedDict):
    messages: Annotated[list, operator.add]   # Accumulate via concatenation
    count: int                                # Plain field -- replacement

def append_hello(state: AccumulatingState) -> dict:
    """Node 1: Appends a greeting to messages."""
    return {"messages": ["Hello"], "count": state["count"] + 1}

def append_world(state: AccumulatingState) -> dict:
    """Node 2: Appends a word to messages."""
    return {"messages": ["World"], "count": state["count"] + 1}

def append_exclamation(state: AccumulatingState) -> dict:
    """Node 3: Appends punctuation to messages."""
    return {"messages": ["!"], "count": state["count"] + 1}

# Build a 3-node graph that accumulates messages
graph = StateGraph(AccumulatingState)
graph.add_node("hello", append_hello)
graph.add_node("world", append_world)
graph.add_node("bang", append_exclamation)
graph.add_edge(START, "hello")
graph.add_edge("hello", "world")
graph.add_edge("world", "bang")
graph.add_edge("bang", END)

app_accum = graph.compile()

# Run it -- messages ACCUMULATE across all three nodes
result = app_accum.invoke({"messages": [], "count": 0})
print("After accumulation (with reducer):")
print("  Messages:", result["messages"])
print("  Count:", result["count"])
print("  All three nodes' messages are concatenated!")


### Part 3: Side-by-Side Comparison


In [ ]:
# Let us visualize the difference between replacement and accumulation
# on the same graph structure.

from IPython.display import Image, display

try:
    display(Image(app_accum.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph structure:")
    print("  START -> hello -> world -> bang -> END")
    print("  (messages field uses operator.add reducer)")

print()
print("Key difference:")
print("  Without reducer: messages = ['!']           (last write wins)")
print("  With reducer:    messages = ['Hello', 'World', '!'] (all appended)")


### Part 4: Real-World Pattern -- Chat Messages


In [ ]:
# The most common use of reducers is maintaining a chat message history.
# LangGraph provides a built-in reducer `add_messages` for this purpose.
# It handles Message objects with proper deduplication by ID.

from langchain_core.messages import HumanMessage, AIMessage   # Message types
from langgraph.graph.message import add_messages              # Built-in reducer

# Define a chat state using the built-in add_messages reducer
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # Accumulate chat messages

def ask_question(state: ChatState) -> dict:
    """Node: Add a user question to the conversation."""
    return {"messages": [HumanMessage(content="What is machine learning?")]}

def answer(state: ChatState) -> dict:
    """Node: Add an AI answer to the conversation."""
    return {"messages": [AIMessage(content="Machine learning is a subset of AI that learns from data.")]}

def followup(state: ChatState) -> dict:
    """Node: Add a follow-up user message."""
    return {"messages": [HumanMessage(content="Can you give an example?")]}

# Build the chat graph
chat_graph = StateGraph(ChatState)
chat_graph.add_node("ask", ask_question)
chat_graph.add_node("answer", answer)
chat_graph.add_node("followup", followup)
chat_graph.add_edge(START, "ask")
chat_graph.add_edge("ask", "answer")
chat_graph.add_edge("answer", "followup")
chat_graph.add_edge("followup", END)

app_chat = chat_graph.compile()

# Run the conversation
chat_result = app_chat.invoke({"messages": []})

print("Conversation after 3 nodes:")
for msg in chat_result["messages"]:
    role = type(msg).__name__               # HumanMessage or AIMessage
    print(f"  {role}: {msg.content}")


### Part 5: Reducer Rules Summary


In [ ]:
#
# **Replacement (default)**
# - Node returns `{"field": new_value}` -> field becomes new_value
# - Old value is discarded
# - Good for: scalar fields, overwrite-on-each-step fields
#
# **Accumulation (with reducer)**
# - Declare: `field: Annotated[Type, reducer_fn]`
# - Node returns `{"field": delta}` -> field = reducer_fn(old, delta)
# - With `operator.add`: lists are concatenated
# - With `add_messages`: chat messages are appended with dedup
#
# **How it works internally**
# 1. Node returns partial update dict
# 2. For each field in the update:
#    - If the field has a reducer: call reducer(old_value, new_value)
#    - If no reducer: replace old_value with new_value
# 3. Merged state becomes the new current state
#
# **Common reducers**
# - `operator.add` -- concatenate lists or add numbers
# - `add_messages` -- append chat messages with ID-based dedup
# - Custom: any `(old, new) -> combined` function

print("State reducers mastered: replacement vs accumulation.")
print("Use Annotated[list, operator.add] for simple list accumulation.")
print("Use Annotated[list, add_messages] for chat message histories.")
